In [ ]:
# 1) Drive + secrets (Colab sidebar 🔑: add KAGGLE_API_TOKEN and GH_TOKEN, enable notebook access)
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['DRV'] = DRV = '/content/drive/MyDrive/s6e10'
os.environ['S6_DATA'] = '/content/data'           # fast local disk
os.environ['S6_ART'] = f'{DRV}/artifacts'         # oof/, test/, ckpt/, experiments.csv live on Drive -> resumable
os.makedirs(os.environ['S6_ART'], exist_ok=True)

In [ ]:
# 2) Code
!test -d /content/s6e10 || git clone -q https://$GH_TOKEN@github.com/0-Playor-0/s6e10-airline.git /content/s6e10
!cd /content/s6e10 && git pull -q && git log -1 --oneline
!pip install -q -U kaggle pyyaml lightgbm

In [ ]:
%%bash
# 3) Data: cached on Drive after the first run, then copied to local disk
set -e
C=$DRV/cache; mkdir -p $C/comp $C/orig /content/data
[ -f $C/comp/train.csv ] || (kaggle competitions download -c playground-series-s6e10 -p $C/comp -q && cd $C/comp && unzip -oq '*.zip' && rm -f *.zip)
[ -f $C/orig/train.csv ] || kaggle datasets download teejmahal20/airline-passenger-satisfaction -p $C/orig -q --unzip
bash /content/s6e10/scripts/fetch_oof.sh $C/public_oof
rsync -a $C/comp $C/orig $C/public_oof /content/data/
du -sh /content/data/*

In [ ]:
# 4) Phase-0 checks (pandas only, ~1 min)
!cd /content/s6e10 && python scripts/verify.py

In [ ]:
# 5) E1a (all kept) then E1b (minus members disclosed as early-stopped on their scored fold); CPU, ~1-1.5 h each.
#    Rerun after a disconnect: cached members + finished folds are skipped. Then the KS lines, the E1a/E1b pick,
#    and ONE validated submission file for the honest stack (written to Drive, NOT submitted).
!cd /content/s6e10 && python src/run.py --exp e1a_stack && python src/run.py --exp e1b_stack
!cd /content/s6e10 && python scripts/make_sub.py --pick-e1

In [ ]:
# 6) E2 own LightGBM (CPU): route profile + original-data teacher + fold-safe TE, our folds; ~1 h, resumable per fold
!cd /content/s6e10 && python src/run.py --exp e2_lgb

In [ ]:
# 7) Does E2 earn a place in the stack? (keep only if nested-CV gain >= 0.00003 over the honest stack)
#    configs/e1c_stack_e2.yaml inherits from e1b_stack; edit `base:` to e1a_stack first if cell 5 picked E1a.
!cd /content/s6e10 && python src/run.py --exp e1c_stack_e2
!grep -E '^(e1a_stack|e1b_stack|e1c_stack_e2|e2_lgb),' $S6_ART/experiments.csv | cut -d, -f1,4,5